# Team 3: Analytics - Full Market Summary in Spark
Full analysis of `data/clean/cleaned_market_data.csv` for Zehra.

Your Spark work must include:
at least six Spark SQL queries
grouped analytics with GROUP BY
sorted results with ORDER BY
at least one time-based query using trade_hour or trade_date
a saved final summary file: results/spark_market_summary.csv
spark.stop() at the end

Your Spark results should connect to the small pandas sample analysis from Team 2, but they must go further. Team 3 should include at least three analyses that are not part of the Team 2 sample check:
volatility ranking
activity ranking
time-based activity analysis
final ranked market summary

In [40]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, hour, date_format, to_date, avg, stddev, min, max, rank, desc, sum, count
from pyspark.sql.window import Window

In [41]:
# EASY 

# 1. Start a Spark session, load the full data/clean/cleaned_market_data.csv file, and print the schema, row count, and column names. 
# This must be the cleaned file from Team 2, not the messy file and not the 50-record pandas sample. 
# Use this to prove that Spark has loaded the dataset correctly. 
# 3 marks

# Example output:

# Spark session started
spark = SparkSession.builder \
    .appName("MarketDataAnalytics") \
    .master("local[*]") \
    .getOrCreate()
print("Spark session started")
# Loaded file: data/clean/cleaned_market_data.csv
file_path = "../data/clean/cleaned_market_data.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)
print(f"Loaded file: {file_path}")
# Row count: 8364
# Columns: symbol, interval, open_time, open, high, low, close, volume, trade_count
# open_time: timestamp
# close: double
# volume: double
print(f"Row count: {df.count()}")
print(f"Columns: {', '.join(df.columns)}")
df.printSchema()


Spark session started
Loaded file: ../data/clean/cleaned_market_data.csv
Row count: 8364
Columns: symbol, open_time, open, high, low, close, volume, close_time, price_range, price_change, percent_change, candle_direction
root
 |-- symbol: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- price_range: double (nullable = true)
 |-- price_change: double (nullable = true)
 |-- percent_change: double (nullable = true)
 |-- candle_direction: string (nullable = true)



In [42]:
# EASY 

# 2. Register the DataFrame as a temporary SQL view named market_data and run a small test query. 
# This lets you run Spark SQL queries against the dataset using the table name market_data. 
# 3 marks

# Example output:

# Temporary SQL view created: market_data
df.createOrReplaceTempView("market_data")
print(f"Temporary SQL view created: market_data")
# Test query returned rows: 10
test_df = spark.sql("SELECT * FROM market_data LIMIT 10")
print(f"Test query returned rows: {test_df.count()}")

Temporary SQL view created: market_data
Test query returned rows: 10


In [43]:

# MEDIUM 

# 3. Add or verify price_range, price_change, percent_change, and candle_direction in Spark. 
# Use the same meanings as Team 2: 
    # price_range = high - low
    # price_change = close - open
    # percent_change = (price_change / open) * 100
    # candle_direction is up, down, or flat based on whether the close price is higher, lower, or equal to the open price. 
# Show a small preview proving the columns exist. 
# 4 marks

# Example output:

# Created/verified columns:
# price_range, price_change, percent_change, candle_direction
print("Created/verified columns:")
df = df.withColumn("price_range", col("high") - col("low")) \
       .withColumn("price_change", col("close") - col("open")) \
       .withColumn("percent_change", (col("price_change") / col("open")) * 100) \
       .withColumn("candle_direction", when(col("close") > col("open"), "up")
                    .when(col("close") < col("open"), "down")
                    .otherwise("flat"))
print("price_range, price_change, percent_change, candle_direction")    

# Example row:
# symbol=BTCUSDT price_range=880.85 price_change=403.87 percent_change=0.51 candle_direction=up
print("\nExample row: ")    
df.select("symbol", "price_range", "price_change", "percent_change", "candle_direction").show(1)



Created/verified columns:
price_range, price_change, percent_change, candle_direction

Example row: 
+--------+-------------------+-------------------+------------------+----------------+
|  symbol|        price_range|       price_change|    percent_change|candle_direction|
+--------+-------------------+-------------------+------------------+----------------+
|AVAXUSDT|0.08300000000000018|0.05599999999999916|0.6447898675877853|              up|
+--------+-------------------+-------------------+------------------+----------------+
only showing top 1 row


In [44]:

# MEDIUM 

# 4. Create time features from open_time, such as trade_date, trade_hour, and day of week. 
# Then run full-dataset Spark SQL queries for average close price by symbol, average volume by symbol, and row count by symbol. 
# Briefly compare this full result with the 50-record pandas sample from Team 2 and explain why the full Spark result is more reliable. 
# 5 marks

# Example output:

# Created time features:
# trade_date, trade_hour, day_of_week
print("Created time features:")
df = df.withColumn("trade_date", to_date(col("open_time"))) \
       .withColumn("trade_hour", hour(col("open_time"))) \
       .withColumn("day_of_week", date_format(col("open_time"), "E"))
print("trade_date, trade_hour, day_of_week")

# Example row:
# open_time=2026-05-04 02:00:00 trade_date=2026-05-04 trade_hour=2 day_of_week=Mon
print("\nExample row: ")
df.select("open_time", "trade_date", "trade_hour", "day_of_week").show(1)

# Average close by symbol:
# BTCUSDT 79160.87
# ETHUSDT 1986.66
print("\nAverage close by symbol:")
avg_close_df = df.groupBy("symbol").avg("close")
avg_close_df.show()

#Average volume by symbol:
print("\nAverage volume by symbol:")
avg_volume_df = df.groupBy("symbol").avg("volume")
avg_volume_df.show()

# Row count by symbol:
# BTCUSDT 842
# ETHUSDT 836
print("Row count by symbol:")
row_count_df = df.groupBy("symbol").count()
row_count_df.show()

# Full Spark result uses all cleaned rows, not only 50 sample rows.

print("Full Spark result uses all cleaned rows, not only 50 sample rows.")
print("\nComparison Analysis:")
print("The full Spark dataset uses all cleaned rows to accurately capture overall volume trends and cyclical price movements across the entire timeline, " \
"whereas Team 2's 50-row Pandas sample introduces severe sampling bias that misses broader market trends, making the complete Spark analytics far more statistically reliable.")

Created time features:
trade_date, trade_hour, day_of_week

Example row: 


+-------------------+----------+----------+-----------+
|          open_time|trade_date|trade_hour|day_of_week|
+-------------------+----------+----------+-----------+
|2026-06-02 10:00:00|2026-06-02|        10|        Tue|
+-------------------+----------+----------+-----------+
only showing top 1 row

Average close by symbol:
+--------+-------------------+
|  symbol|         avg(close)|
+--------+-------------------+
| BTCUSDT|   74944.7145199064|
| BNBUSDT|  642.0373411764717|
|DOGEUSDT|0.10248355871886115|
| DOTUSDT| 1.2124775757575759|
|LINKUSDT|  9.291368544600935|
| SOLUSDT|  82.68032941176479|
| ADAUSDT|0.23738455971049474|
| XRPUSDT| 1.3395705596107061|
|AVAXUSDT|  8.919383211678827|
| ETHUSDT| 2091.4827417380698|
+--------+-------------------+


Average volume by symbol:
+--------+-------------------+
|  symbol|        avg(volume)|
+--------+-------------------+
| BTCUSDT|  716.0706513817332|
| BNBUSDT| 7763.2758388235225|
|DOGEUSDT|3.665896605812574E7|
| DOTUSDT| 285977.92727

In [45]:

# HARD

# 5. Create a volatility ranking by symbol using average, minimum, maximum, and standard deviation of price_range. 
# A symbol with a higher average or standard deviation of price_range is usually more volatile. 
# 4 marks

# Example output:

# Volatility ranking:
# rank=1 symbol=BTCUSDT avg_price_range=842.21 stddev_price_range=410.32
# rank=2 symbol=ETHUSDT avg_price_range=26.73 stddev_price_range=12.84

print("Volatility ranking:")
volatility_df = df.groupBy("symbol") \
    .agg(
        avg(col("price_range")).alias("avg_price_range"),
        min(col("price_range")).alias("min_price_range"),
        max(col("price_range")).alias("max_price_range"),
        stddev(col("price_range")).alias("stddev_price_range")
    )
windowSpec = Window.orderBy(desc("stddev_price_range"))
final_ranked_df = volatility_df.withColumn("rank", rank().over(windowSpec))
final_ranked_df.select("rank", "symbol", "avg_price_range", "min_price_range", "max_price_range", "stddev_price_range").show()

Volatility ranking:
+----+--------+--------------------+--------------------+--------------------+--------------------+
|rank|  symbol|     avg_price_range|     min_price_range|     max_price_range|  stddev_price_range|
+----+--------+--------------------+--------------------+--------------------+--------------------+
|   1| BTCUSDT|   420.8679976580788|   67.07000000000698|  2239.1399999999994|   294.6457525384541|
|   2| ETHUSDT|  15.279645042839624|                2.25|   92.88000000000011|   11.21384834052267|
|   3| BNBUSDT|    4.44508235294118|  0.5900000000000318|   40.99000000000001|   3.530181335868877|
|   4| SOLUSDT|   0.726776470588235| 0.13999999999998636|  3.6700000000000017|  0.4756211827258044|
|   5|AVAXUSDT| 0.08846836982968367|0.018000000000000682|  0.5200000000000014| 0.05431527800373586|
|   6|LINKUSDT| 0.08945892018779333|0.013999999999999346|  0.4399999999999995|   0.052453678991332|
|   7| DOTUSDT|0.013910303030303018|0.002000000000000...| 0.07400000000000007|0.

In [46]:
# HARD

# 6. Create an activity ranking by symbol using total trades, total quote volume, and average volume. 
# A symbol with more trades and higher quote volume should appear more active. 

# 4 marks

# Example output:

# Activity ranking:
# rank=1 symbol=XRPUSDT total_trades=80311880 total_quote_volume=912345678.22
# rank=2 symbol=BTCUSDT total_trades=73422311 total_quote_volume=812345678.44
print("Activity ranking:")
activity_df = df.groupBy("symbol") \
    .agg(
        count(col("volume")).alias("total_trades"),
        sum(col("volume") * col("close")).alias("total_quote_volume"),
        avg(col("volume")).alias("average_volume")
    )
windowSpec = Window.orderBy(desc("total_quote_volume"))
final_ranked_df = activity_df.withColumn("rank", rank().over(windowSpec))
final_ranked_df.select("rank", "symbol", "total_trades", "total_quote_volume", "average_volume").show()

Activity ranking:
+----+--------+------------+--------------------+-------------------+
|rank|  symbol|total_trades|  total_quote_volume|     average_volume|
+----+--------+------------+--------------------+-------------------+
|   1| BTCUSDT|         854|4.450733476749984E10|  716.0706513817332|
|   2| ETHUSDT|         817|2.262032077501259E10| 13738.828001958385|
|   3| SOLUSDT|         850| 7.574862494201085E9| 110026.02972235296|
|   4| XRPUSDT|         822| 4.423026222955077E9|   4064797.52858881|
|   5| BNBUSDT|         850|4.2729785639239917E9| 7763.2758388235225|
|   6|DOGEUSDT|         843|3.1785144311233444E9|3.665896605812574E7|
|   7| ADAUSDT|         829|  1.04203609907945E9|   5717135.14933655|
|   8|LINKUSDT|         852| 9.642718223690606E8| 122629.82917840383|
|   9|AVAXUSDT|         822| 6.590745745332594E8|  92451.02929440384|
|  10| DOTUSDT|         825|   2.8265764561734E8| 285977.92727272736|
+----+--------+------------+--------------------+-------------------+



In [47]:

# HARD

# 7. Analyze activity by time using the time features. 
# Find the busiest hour and busiest date by trade_count and quote_volume, 
# then explain when market activity was highest in the cleaned dataset. 
# 4 marks

# Example output:

# Busiest hour by trades:
# trade_hour=14 total_trades=2834412
print("Busiest hour by trades:")
busiest_hour_df = df.groupBy("trade_hour").count().orderBy(desc("count"))
busiest_hour_df.show(1) 

# Busiest date by quote volume:
# trade_date=2026-05-04 total_quote_volume=734567890.55
print("\nBusiest date by quote volume:")
busiest_date_df = df.groupBy("trade_date") \
    .agg(sum(col("volume") * col("close")).alias("total_quote_volume"))
busiest_date_df = busiest_date_df.orderBy(desc("total_quote_volume"))
busiest_date_df.show(1)

Busiest hour by trades:
+----------+-----+
|trade_hour|count|
+----------+-----+
|         9|  366|
+----------+-----+
only showing top 1 row

Busiest date by quote volume:
+----------+-------------------+
|trade_date| total_quote_volume|
+----------+-------------------+
|2026-06-05|5.861730500690326E9|
+----------+-------------------+
only showing top 1 row


In [ ]:
# HARD

# 8. Build a final ranked market summary table with one row per symbol. Include total records, average volume, total trades, average percent change, average price range, up/down/flat candle counts, volatility rank, activity rank, and a short interpretation for Zehra. The final table should help Zehra quickly see which symbols were most active, most volatile, and most important to mention in the report. Save this result as results/spark_market_summary.csv. 8 marks

# Example output:

# Final ranked market summary created
# Rows in summary: 10
# Saved: results/spark_market_summary.csv
print("Final ranked market summary created")
summary_base = df.groupBy("symbol") \
    .agg(
        count("*").alias("total_records"),
        avg(col("volume")).alias("average_volume"),
        count(col("volume")).alias("total_trades"),
        avg(col("percent_change")).alias("average_percent_change"),
        avg(col("price_range")).alias("average_price_range"),
        sum(when(col("candle_direction") == "up", 1).otherwise(0)).alias("up_candle_count"),
        sum(when(col("candle_direction") == "down", 1).otherwise(0)).alias("down_candle_count"),
        sum(when(col("candle_direction") == "flat", 1).otherwise(0)).alias("flat_candle_count")
    )
vol_window = Window.orderBy(desc("average_price_range"))
act_window = Window.orderBy(desc("total_trades"))
summary_df = summary_base \
    .withColumn("volatility_rank", rank().over(vol_window)) \
    .withColumn("activity_rank", rank().over(act_window))
print("Rows in summary: {}".format(summary_df.count()))
summary_df.write.mode("overwrite").csv("../results/spark_market_summary.csv", header=True)
print("Saved: results/spark_market_summary.csv")

# Top activity symbol: XRPUSDT
# Top volatility symbol: BTCUSDT
# Interpretation: XRPUSDT had the highest trading activity, while BTCUSDT had the widest average price range.
print ("Top activity symbol: {}".format(summary_df.orderBy(desc("total_trades")).first()["symbol"]))
print ("Top volatility symbol: {}".format(summary_df.orderBy(desc("average_price_range")).first()["symbol"]))
print ("Interpretation: BTCUSDT had the highest trading activity and the widest average price range, making it the most active and most volatile symbol in the dataset.")

# Stop the Spark Session cleanly
spark.stop()
print("Spark Session stopped successfully.")

Final ranked market summary created
Rows in summary: 10
Rows in summary: 10
Saved: results/spark_market_summary.csv
Top activity symbol: BTCUSDT
Top volatility symbol: BTCUSDT
Interpretation: BTCUSDT had the highest trading activity and the widest average price range, making it the most active and most volatile symbol in the dataset.
Spark Session stopped successfully.
